# 24 Writing Data

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Write DataFrames to files and tables with the right <b>save mode</b>, format and target (path vs managed table), control how many output files you create, and avoid the classic traps: positional <code>insertInto</code>, accidental full overwrites, and the small-files problem.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Every pipeline ends with a write, and the write decides whether the result is correct (idempotent, no duplicates, nothing lost) and fast to read later (sensible file sizes). On Databricks the default target is a <b>Delta table</b> registered in Unity Catalog. "What are the save modes?" and "Why do I get thousands of small files?" are standard interview questions.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 The pieces of a write</b><br>
```python<br>df.write.format("delta").mode("overwrite").option(...).partitionBy(...).saveAsTable("catalog.schema.table")<br><code>`</code>
<ul><li><b>Format:</b> <code>delta</code> (Databricks default for tables), <code>parquet</code>, <code>csv</code>, <code>json</code>, ...</li><li><b>Mode:</b> <code>errorifexists</code> (default), <code>append</code>, <code>overwrite</code>, <code>ignore</code></li><li><b>Target:</b> <code>.save(path)</code> writes files to a location. <code>.saveAsTable(name)</code> writes <b>and</b> registers a table in the catalog</li><li><b>Output files:</b> one file (or more) per <b>partition of the DataFrame</b> being written, per output partition folder</li><li><b>DataFrameWriterV2:</b> <code>df.writeTo("table").append() / .overwritePartitions() / .createOrReplace()</code>, an explicit, table-first API</li></ul>
</div>

| Mode | Target doesn't exist | Target exists |
|---|---|---|
| `errorifexists` / `error` (default) | Create | **Fail** |
| `append` | Create | Add rows |
| `overwrite` | Create | **Replace** all data |
| `ignore` | Create | Do nothing, silently |

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A nightly job recomputes the <code>daily_sales</code> Gold table for yesterday. Version 1 used <code>mode("append")</code>, so a re-run after a failure <b>duplicated</b> a day of revenue. Version 2 used <code>mode("overwrite")</code> and wiped two years of history. Version 3 overwrites <b>only the affected date partition</b> (dynamic partition overwrite, or Delta's <code>replaceWhere</code>), which makes the job idempotent: running it twice gives the same result.
</div>

## Syntax

```python
# Files at a path (for example a volume)
df.write.mode("overwrite").parquet("/Volumes/workspace/spark_basics/raw_files/out/sales")

# Managed table in Unity Catalog (Delta by default on Databricks)
df.write.mode("overwrite").saveAsTable("workspace.spark_basics.sales")
df.write.mode("append").saveAsTable("workspace.spark_basics.sales")

# Table-first API (DataFrameWriterV2)
df.writeTo("workspace.spark_basics.sales").createOrReplace()

# SQL
spark.sql("CREATE OR REPLACE TABLE workspace.spark_basics.sales_copy AS SELECT * FROM workspace.spark_basics.sales")
```

## Setup: schema, volume and sample data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates the schema and volume, defines where this lesson's files and tables go, and builds a small sales DataFrame.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The <code>BASE</code> path and table prefix, then a 6-row sales table.
</div>

In [0]:
CATALOG, SCHEMA, VOLUME = "workspace", "spark_basics", "raw_files"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/lesson24"
DB = f"{CATALOG}.{SCHEMA}"            # tables go to workspace.spark_basics
print("BASE =", BASE, "| tables in", DB)

In [0]:
import os
from pyspark.sql import functions as F

sales = spark.createDataFrame(
    [
        (1, "2024-10-01", "IN", 120.0), (2, "2024-10-01", "UK", 80.0), (3, "2024-10-02", "IN", 300.0),
        (4, "2024-10-02", "AE", 45.0), (5, "2024-10-03", "UK", 99.0), (6, "2024-10-03", "IN", 15.0),
    ],
    "sale_id INT, sale_date STRING, country STRING, amount DOUBLE",
).withColumn("sale_date", F.to_date("sale_date"))
sales.show()

def list_data_files(path):
    """Data files Spark wrote in a folder (ignores markers like _SUCCESS and hidden checksum files)."""
    return sorted(n for n in os.listdir(path) if not n.startswith((".", "_")))

## 1. Save modes on a path

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes the same data to a folder four times with different modes and counts the rows after each write.<br><br>
<b>Why it matters</b><br>The default mode <b>fails</b> if the target exists, which protects you from accidents. <code>append</code> adds duplicates when a job is re-run. <code>overwrite</code> replaces everything. <code>ignore</code> silently skips the write, which can hide bugs.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
After the first write: 6 rows. <code>errorifexists</code> then fails with a "path already exists" error. <code>append</code>: 12 rows. <code>ignore</code>: still 12. <code>overwrite</code>: back to 6.
</div>

In [0]:
path = f"{BASE}/modes/sales"
count = lambda: spark.read.parquet(path).count()

sales.write.mode("overwrite").parquet(path)
print("first write          :", count())

try:
    sales.write.parquet(path)                      # default mode = errorifexists
except Exception as e:
    print("errorifexists        :", type(e).__name__, "-", str(e).splitlines()[0][:90])

sales.write.mode("append").parquet(path)
print("after append         :", count())

sales.write.mode("ignore").parquet(path)
print("after ignore         :", count())

sales.write.mode("overwrite").parquet(path)
print("after overwrite      :", count())

## 2. How many files does a write create?

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes 10,000 generated rows with 1, 4 and 8 DataFrame partitions and counts the output files.<br><br>
<b>Why it matters</b><br>Each write task produces its own file, so the number of files follows the number of <b>DataFrame partitions</b>. A DataFrame with 2,000 tiny partitions writes 2,000 tiny files: the <b>small-files problem</b>, which makes every later read slow. Control it with <code>repartition</code> or <code>coalesce</code> before writing, or <code>maxRecordsPerFile</code> to cap file size.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
1, 4 and 8 data files respectively, then 3 files when <code>maxRecordsPerFile=2</code> splits the 6 sales rows. One detail: an <b>empty</b> partition writes no file, so 6 rows spread over 8 partitions would give fewer than 8 files.
</div>

In [0]:
rows = spark.range(10_000)
for n in [1, 4, 8]:
    out = f"{BASE}/files/n{n}"
    rows.repartition(n).write.mode("overwrite").parquet(out)
    print(f"{n} partitions -> {len(list_data_files(out))} data files")

out = f"{BASE}/files/capped"
sales.coalesce(1).write.mode("overwrite").option("maxRecordsPerFile", 2).parquet(out)
print("maxRecordsPerFile=2 ->", len(list_data_files(out)), "data files")

## 3. Other file formats

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes CSV (with a header) and JSON, then reads them back.<br><br>
<b>Why it matters</b><br>Exports for other systems are often CSV or JSON. Remember: CSV loses types, and without <code>header=True</code> it loses column names too. Even <code>coalesce(1)</code> still produces a <b>folder</b> with one part file inside, not a single named file.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A folder containing one <code>part-...csv</code> file, then the data read back from CSV and from JSON.
</div>

In [0]:
sales.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{BASE}/export/sales_csv")
print(list_data_files(f"{BASE}/export/sales_csv"))
spark.read.option("header", True).csv(f"{BASE}/export/sales_csv").show(3)

sales.write.mode("overwrite").json(f"{BASE}/export/sales_json")
spark.read.json(f"{BASE}/export/sales_json").show(3)

## 4. Managed tables with `saveAsTable`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Saves the DataFrame as a managed table, appends to it, inspects it, and reads it back with <code>spark.table</code> and SQL.<br><br>
<b>Why it matters</b><br>Tables are how data is shared, governed and queried in Databricks. A <b>managed</b> table means Unity Catalog manages both the metadata and the storage location. You refer to it by name, never by path. On Databricks the default format is <b>Delta</b>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
6 rows, then 12 after the append. <code>DESCRIBE EXTENDED</code> shows the columns plus details such as <code>Type: MANAGED</code> and the provider (<code>delta</code> on Databricks).
</div>

In [0]:
table = f"{DB}.lesson24_sales"

sales.write.mode("overwrite").saveAsTable(table)
print("rows:", spark.table(table).count())

sales.write.mode("append").saveAsTable(table)
print("rows after append:", spark.sql(f"SELECT count(*) FROM {table}").first()[0])

spark.sql(f"DESCRIBE EXTENDED {table}").filter("col_name IN ('Type', 'Provider', 'Location', 'Catalog', 'Database')").show(truncate=False)

## 5. Inspect a Delta table: `DESCRIBE DETAIL` and history

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows the number of files, size and format of the table, and the history of writes.<br><br>
<b>Why it matters</b><br>Because Delta keeps a transaction log, every write is recorded. <code>overwrite</code> on a Delta table doesn't destroy the old files immediately, so you can see (and even query) previous versions. That is time travel, covered in <code>03_delta_lake</code>.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Delta commands such as <code>DESCRIBE DETAIL</code> and <code>DESCRIBE HISTORY</code> work in Free Edition. They need Delta, so they don't run in plain open-source Spark.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>format = delta</code>, <code>numFiles</code> and <code>sizeInBytes</code>. The history shows a <code>CREATE OR REPLACE TABLE AS SELECT</code> (or <code>WRITE</code>) operation followed by a <code>WRITE</code> with mode <code>Append</code>.
</div>

In [0]:
display(spark.sql(f"DESCRIBE DETAIL {table}").select("format", "numFiles", "sizeInBytes", "location"))
display(spark.sql(f"DESCRIBE HISTORY {table}").select("version", "timestamp", "operation", "operationParameters"))

## 6. The `insertInto` trap: positions, not names

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Appends a DataFrame whose columns are in a different order, first with <code>insertInto</code> and then with <code>saveAsTable</code> in append mode.<br><br>
<b>Why it matters</b><br><code>insertInto</code> matches columns <b>by position</b>, like <code>union</code>. If the column order differs, values land in the wrong columns, or the write fails on a type mismatch. <code>saveAsTable(..., mode="append")</code> matches by name. This bug has corrupted real production tables.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>insertInto</code> fails with a cast error (the date string can't become an integer) or writes values into the wrong columns. The name-based append succeeds, and the new row has every value in the correct column.
</div>

In [0]:
reordered = spark.createDataFrame([(99.5, "AE", "2024-10-04", 7)], "amount DOUBLE, country STRING, sale_date STRING, sale_id INT") \
    .withColumn("sale_date", F.to_date("sale_date"))

try:
    reordered.write.insertInto(table)
    print("insertInto succeeded. Check the data: values may be in the wrong columns!")
except Exception as e:
    print("insertInto failed:", type(e).__name__, "-", str(e).splitlines()[0][:120])

reordered.write.mode("append").saveAsTable(table)
spark.table(table).filter("sale_id = 7").show()

## 7. Overwrite only some data: dynamic partition overwrite and `replaceWhere`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes a table partitioned by <code>sale_date</code>, then re-processes <b>one day</b> and replaces only that day.<br><br>
<b>Why it matters</b><br>This is how idempotent daily jobs work. A plain <code>overwrite</code> would delete every other day. Two ways to replace part of a table:
<ul><li><code>partitionOverwriteMode=dynamic</code>: replace only the partitions present in the new data (works for partitioned Parquet and Delta)</li><li><code>replaceWhere</code> (Delta): replace exactly the rows matching a condition, and fail if the new data doesn't match it</li></ul>
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
3 days and 6 rows initially. After the dynamic overwrite of 2024-10-02: still 3 days, and that day has 1 corrected row (amount 999.0). Other days are untouched, so the total is 5 rows.
</div>

In [0]:
part_table = f"{DB}.lesson24_sales_by_day"
sales.write.mode("overwrite").partitionBy("sale_date").saveAsTable(part_table)

corrected_day = spark.createDataFrame([(3, "2024-10-02", "IN", 999.0)], "sale_id INT, sale_date STRING, country STRING, amount DOUBLE") \
    .withColumn("sale_date", F.to_date("sale_date"))

(
    corrected_day.write
    .mode("overwrite")
    .option("partitionOverwriteMode", "dynamic")
    .saveAsTable(part_table)
)
spark.table(part_table).groupBy("sale_date").agg(F.count("*").alias("rows"), F.sum("amount").alias("amount")).orderBy("sale_date").show()

In [0]:
# Delta-only alternative: replace exactly the rows that match a predicate.
(
    corrected_day.write
    .mode("overwrite")
    .option("replaceWhere", "sale_date = '2024-10-02'")
    .saveAsTable(part_table)
)
display(spark.table(part_table).orderBy("sale_date"))

## 8. The table-first API and SQL

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses <code>writeTo(...)</code> (DataFrameWriterV2) and SQL <code>CREATE TABLE AS SELECT</code> / <code>INSERT INTO</code>.<br><br>
<b>Why it matters</b><br><code>writeTo</code> makes the intent explicit (<code>create</code>, <code>createOrReplace</code>, <code>append</code>, <code>overwritePartitions</code>) and matches columns by name. Many Databricks teams write tables purely in SQL, and SQL syntax is common on the exam.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A table created with <code>createOrReplace</code> containing the UK rows, then 3 rows after an <code>INSERT INTO ... SELECT</code> in SQL.
</div>

In [0]:
uk_table = f"{DB}.lesson24_uk_sales"
sales.filter("country = 'UK'").writeTo(uk_table).createOrReplace()
print("after createOrReplace:", spark.table(uk_table).count())

spark.sql(f"INSERT INTO {uk_table} SELECT 100, date'2024-10-05', 'UK', 55.0")
spark.sql(f"SELECT * FROM {uk_table} ORDER BY sale_id").show()

spark.sql(f"CREATE OR REPLACE TABLE {DB}.lesson24_in_sales AS SELECT * FROM {table} WHERE country = 'IN'")
print("CTAS rows:", spark.table(f"{DB}.lesson24_in_sales").count())

## Under the hood

```
df.write.mode("overwrite").parquet(path)        (action: starts a job immediately)

1. Driver: check mode, prepare the output committer
2. One task per DataFrame partition:
      write rows to a temporary file  ->  part-00000-<uuid>.snappy.parquet
3. Job commit: temporary files are moved/committed, then a _SUCCESS marker is written
   (Delta instead writes a JSON commit file in _delta_log, which makes the write atomic)

Files per write  =  number of DataFrame partitions  x  output partition folders they touch
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> `write`, `save` and `saveAsTable` trigger a job immediately, unlike transformations.

**Delta vs plain files:** with plain Parquet, a failed job can leave partial files, and readers can see half-written data. Delta writes are **atomic**: a commit either appears in the log or it doesn't, so readers never see partial results.

Check the job in the Spark UI (or the query profile on serverless). The number of write tasks equals the number of partitions. Run the cell below to see how partitions map to files.

In [0]:
demo = spark.range(1_000_000).repartition(5)
out = f"{BASE}/files/demo"
demo.write.mode("overwrite").parquet(out)
print("partitions:", demo.select(F.spark_partition_id()).distinct().count(), "| files:", len(list_data_files(out)))

In [0]:
# Optional cleanup: remove this lesson's tables and files.
for t in ["lesson24_sales", "lesson24_sales_by_day", "lesson24_uk_sales", "lesson24_in_sales"]:
    spark.sql(f"DROP TABLE IF EXISTS {DB}.{t}")
dbutils.fs.rm(BASE, True)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>[PATH_ALREADY_EXISTS]</code> or <code>TABLE_OR_VIEW_ALREADY_EXISTS</code></b><br>
The default mode is <code>errorifexists</code>. Choose <code>append</code> or <code>overwrite</code> deliberately.<br><br>
<b>⛔ Problem: duplicates after re-running a job</b><br><code>append</code> isn't idempotent. Overwrite the affected partitions, use <code>replaceWhere</code>, or <code>MERGE</code> (Delta lesson 07).<br><br>
<b>⛔ Problem: history wiped out</b><br>A plain <code>overwrite</code> replaced the whole table. Use dynamic partition overwrite or <code>replaceWhere</code>. On Delta, <code>RESTORE</code> can recover (lesson in <code>03_delta_lake</code>).<br><br>
<b>⛔ Problem: values in the wrong columns</b><br><code>insertInto</code> matches by position. Use <code>saveAsTable</code> in append mode, <code>writeTo().append()</code>, or select columns in the table's order.<br><br>
<b>⛔ Problem: thousands of tiny files</b><br>Too many DataFrame partitions at write time. <code>coalesce</code>/<code>repartition</code> first, or use Delta <code>OPTIMIZE</code> and auto-compaction.<br><br>
<b>⛔ Problem: wanted a single CSV file but got a folder</b><br>Spark always writes a folder. Use <code>coalesce(1)</code> for one part file inside it, and only for small data.<br><br>
**⛔ Problem: writes to <code>dbfs:/</code> or <code>/tmp</code> fail on Free Edition**<br>Use a Unity Catalog volume path or a table name.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What are the save modes in Spark?</b><br>
<code>errorifexists</code> (the default) fails if the target exists, <code>append</code> adds data, <code>overwrite</code> replaces it, and <code>ignore</code> does nothing if the target exists.<br><br>

<b>🎤 2. What is the difference between <code>save(path)</code> and <code>saveAsTable(name)</code>?</b><br>
<code>save</code> writes files to a location only. <code>saveAsTable</code> writes the data and registers a table in the metastore or Unity Catalog, so it can be queried by name and governed. On Databricks, <code>saveAsTable</code> creates a managed Delta table by default.<br><br>

<b>🎤 3. Why does Spark produce many small files, and how do you fix it?</b><br>
Each task writes its own file, so the file count follows the number of DataFrame partitions (times the output partitions). Fixes: <code>coalesce</code> or <code>repartition</code> before writing, <code>maxRecordsPerFile</code>, and on Delta, <code>OPTIMIZE</code> or auto-compaction and optimized writes.<br><br>

<b>🎤 4. How do you make a daily batch write idempotent?</b><br>
Overwrite only the data for that day, with dynamic partition overwrite or Delta's <code>replaceWhere</code>, or use <code>MERGE</code> on a key. A plain <code>append</code> duplicates data when the job re-runs.<br><br>

<b>🎤 5. What is the risk of <code>insertInto</code>?</b><br>
It matches columns by position, not by name, so a different column order puts values in the wrong columns or fails. Name-based appends are safer.<br><br>

<b>🎤 6. Is a write a transformation or an action?</b><br>
An action. It triggers a job immediately.<br><br>

<b>🎤 7. What is the difference between <code>coalesce(1)</code> and <code>repartition(1)</code> before writing?</b><br>
Both produce one output file. <code>coalesce(1)</code> avoids a shuffle but can push all upstream work into a single task. <code>repartition(1)</code> adds a shuffle but keeps the upstream work parallel.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A nightly job appends yesterday's data to a Delta table. After a failure, the job is re-run and the data for that day is duplicated. Which change makes the job idempotent while preserving other days?</b><br>
A. Use <code>mode("overwrite")</code><br>
B. Use <code>mode("ignore")</code><br>
C. Use <code>mode("overwrite")</code> with <code>replaceWhere</code> for yesterday's date<br>
D. Use <code>mode("errorifexists")</code><br>
<details><summary><b>Show answer</b></summary><b>C.</b> It replaces only the matching data. A wipes the whole table, B would skip the write entirely, and D fails.</details><br><br>

<b>Q2. Which statement creates a managed table from a query?</b><br>
A. <code>CREATE TABLE t LOCATION '/Volumes/...'</code><br>
B. <code>CREATE OR REPLACE TABLE t AS SELECT * FROM src</code><br>
C. <code>INSERT OVERWRITE DIRECTORY '/Volumes/...' SELECT * FROM src</code><br>
D. <code>CREATE VIEW t AS SELECT * FROM src</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> CTAS without a <code>LOCATION</code> creates a managed table. A specifies a location (external), C writes files, and D creates a view.</details><br><br>

<b>Q3. What is the default save mode of <code>DataFrameWriter</code>?</b><br>
A. <code>append</code><br>
B. <code>overwrite</code><br>
C. <code>errorifexists</code><br>
D. <code>ignore</code><br>
<details><summary><b>Show answer</b></summary><b>C.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Write <code>sales</code> to <code>BASE/practice/sales</code> as JSON with exactly 2 output files, and prove it</li><li>Save <code>sales</code> as a table <code>lesson24_practice</code> partitioned by <code>country</code></li><li>Re-process only the <code>UK</code> partition with one corrected row, keeping IN and AE untouched, and show row counts per country</li><li>Append one row whose columns are in a different order, safely by name</li><li>In a comment, explain why <code>mode("ignore")</code> is risky in production pipelines</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Exactly 2 files
sales.repartition(2).write.mode("overwrite").json(f"{BASE}/practice/sales")
print("files:", len(list_data_files(f"{BASE}/practice/sales")))

# 2. Partitioned table
ptable = f"{DB}.lesson24_practice"
sales.write.mode("overwrite").partitionBy("country").saveAsTable(ptable)

# 3. Replace only the UK partition
uk_fix = spark.createDataFrame([(2, "2024-10-01", "UK", 85.0)], "sale_id INT, sale_date STRING, country STRING, amount DOUBLE") \
    .withColumn("sale_date", F.to_date("sale_date"))
uk_fix.write.mode("overwrite").option("partitionOverwriteMode", "dynamic").saveAsTable(ptable)
spark.table(ptable).groupBy("country").count().orderBy("country").show()

# 4. Name-based append with a different column order
spark.createDataFrame([(5.0, 8, "AE")], "amount DOUBLE, sale_id INT, country STRING") \
    .withColumn("sale_date", F.to_date(F.lit("2024-10-06"))) \
    .write.mode("append").saveAsTable(ptable)
spark.table(ptable).filter("sale_id = 8").show()

# 5. 'ignore' silently skips the write when the target exists, so a job can "succeed"
#    without writing anything, and nobody notices that the data is stale.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Modes: <code>errorifexists</code> (default), <code>append</code>, <code>overwrite</code>, <code>ignore</code></li><li><code>save(path)</code> writes files. <code>saveAsTable(name)</code> writes and registers a (Delta, on Databricks) table</li><li>Writes are actions. One task per partition writes its own file</li><li>Control files with <code>repartition</code>/<code>coalesce</code> and <code>maxRecordsPerFile</code>. Avoid small files</li><li><code>insertInto</code> is positional. Prefer name-based appends</li><li>Idempotent loads: dynamic partition overwrite, <code>replaceWhere</code> (Delta) or <code>MERGE</code></li><li>Delta writes are atomic and recorded in history</li><li>Free Edition: write to volumes (<code>/Volumes/...</code>) or Unity Catalog tables</li></ul>
</div>

| Task | Code |
|---|---|
| Files | `df.write.mode("overwrite").parquet(path)` |
| Table | `df.write.mode("append").saveAsTable("cat.sch.t")` |
| V2 API | `df.writeTo("cat.sch.t").createOrReplace()` |
| Partitioned | `.partitionBy("date")` |
| Replace some partitions | `.option("partitionOverwriteMode", "dynamic")` |
| Replace by predicate | `.option("replaceWhere", "date = '2024-10-02'")` |
| File count | `df.coalesce(n)`, `.option("maxRecordsPerFile", N)` |
| SQL | `CREATE OR REPLACE TABLE t AS SELECT ...`, `INSERT INTO t SELECT ...` |

## Git commit

```
git add 01_spark_basics/24_writing_data.ipynb
git commit -m "add 24 writing data notebook"
```